# Consolidación del Pipeline Final, Persistencia y Predicción Final

Este es el último paso del proyecto. Aquí realizaremos:
1. **Entrenamiento Final:** Entrenaremos el Pipeline definitivo con todos los datos históricos (`X_train_full`).
2. **Evaluación Final en Test:** Evaluaremos el modelo por primera y única vez en el conjunto de prueba descolgado (`X_test_full`), aplicando nuestro umbral óptimo.
3. **Persistencia del Modelo:** Guardaremos el Pipeline completo en un archivo `.pkl` listo para ser desplegado en producción.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

### 1. Carga de Datos (Train y Test Final)

In [ ]:
base_path = '../data/model_input/cv/'

# Datos Históricos (Para entrenar el modelo final)
X_train_full = pd.read_csv(base_path + 'X_train_full.csv')
y_train_full = pd.read_csv(base_path + 'y_train_full.csv').values.ravel()

# Datos de Prueba Descolgados (El "Futuro" Real)
X_test_full = pd.read_csv(base_path + 'X_test.csv')
y_test_full = pd.read_csv(base_path + 'y_test.csv').values.ravel()

print(f"Datos de Entrenamiento (Histórico): {X_train_full.shape[0]} reservas.")
print(f"Datos de Prueba (Futuro): {X_test_full.shape[0]} reservas.")

### 2. Construcción y Entrenamiento del Pipeline Definitivo

In [ ]:
# 1. Preprocesador
vars_categoricas = ['meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type']
vars_numericas = ['lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 'previous_cancellations', 'required_car_parking_spaces', 'total_of_special_requests', 'agent', 'company', 'arrival_date_year', 'arrival_month_num']
vars_binarias = ['is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)

# 2. Hiperparámetros Ganadores (Random Forest)
gb_best_params = {
    'n_estimators': 76,
    'learning_rate': 0.1,
    'max_depth': 5,
    'min_samples_split': 2,
    'min_samples_leaf': 4,
    'random_state': 42
}

# 3. Pipeline Final
pipeline_produccion = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(**gb_best_params))
])

print("Entrenando el modelo definitivo para producción...")
pipeline_produccion.fit(X_train_full, y_train_full)
print("¡Entrenamiento completado!")

### 3. Evaluación Final en el Conjunto de Test
Esta es la métrica real que reportaremos al negocio. Aplicaremos el umbral óptimo descubierto en la fase anterior.

In [ ]:
# REEMPLAZA ESTE VALOR CON EL UMBRAL ÓPTIMO QUE OBTUVISTE EN EL NOTEBOOK 09
UMBRAL_OPTIMO = 0.3928 

# 1. Generar probabilidades en el Test Set
y_proba_test = pipeline_produccion.predict_proba(X_test_full)[:, 1]

# 2. Aplicar el umbral óptimo
y_pred_test_optimo = (y_proba_test >= UMBRAL_OPTIMO).astype(int)

# 3. Evaluar
print("="*50)
print("RENDIMIENTO FINAL EN PRODUCCIÓN (TEST SET)")
print("="*50)
print(f"ROC AUC:  {roc_auc_score(y_test_full, y_proba_test):.4f}")
print(f"Accuracy: {accuracy_score(y_test_full, y_pred_test_optimo):.4f}")
print("-"*50)
print("Reporte de Clasificación (Umbral Ajustado):\n")
print(classification_report(y_test_full, y_pred_test_optimo))
print("="*50)

# Matriz de Confusión
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test_full, y_pred_test_optimo), annot=True, fmt='d', cmap='Greens', cbar=False)
plt.title(f'Matriz de Confusión Final (Umbral {UMBRAL_OPTIMO})', fontsize=14, pad=15)
plt.xlabel('Predicción del Modelo', fontsize=12)
plt.ylabel('Realidad', fontsize=12)
plt.tight_layout()
plt.show()

### 4. Persistencia del Modelo (Guardar para Producción)
Guardaremos el Pipeline completo en un archivo `.pkl`. Este archivo contiene todo lo necesario (escaladores, codificadores y el modelo de Random Forest) para recibir datos crudos y devolver una predicción.

In [ ]:
# Crear directorio si no existe
os.makedirs('../models', exist_ok=True)

ruta_modelo = '../models/hotel_cancellation_gb_pipeline.pkl'

# Guardar el modelo
joblib.dump(pipeline_produccion, ruta_modelo)

print(f"✅ ¡Modelo guardado exitosamente en: {ruta_modelo}!")
print(f"Tamaño del archivo: {os.path.getsize(ruta_modelo) / (1024*1024):.2f} MB")

print("\nPara usar el modelo en producción, simplemente ejecuta:")
print("modelo_cargado = joblib.load('models/hotel_cancellation_gb_pipeline.pkl')")
print(f"probabilidades = modelo_cargado.predict_proba(nuevos_datos)[:, 1]")
print(f"predicciones = (probabilidades >= {UMBRAL_OPTIMO}).astype(int)")